## Progettazione di un data warehouse
Progettare un data warehouse per la gestione delle problematiche illustrate nei punti seguenti relative ai magazzini di una ditta di elettrodomestici italiana con sedi in tutta Italia.

### Descrizione del problema
Una grande ditta di elettrodomestici ha magazzini sparsi in tutta Italia. La dirigenza dell’azienda ha la necessità di analizzare l’attuale uso dei magazzini al fine di capire se e quanto i vari magazzini sono effettivamente utilizzati in funzione della loro dislocazione geografica. La finalità dell’azienda è quella di decidere quali magazzini ampliare, quali ridimensionare e quali chiudere o affittare, anche solo parzialmente,
a terze parti.
La dirigenza della ditta stipula periodicamente delle polizze assicurative per coprire i danni dei furti che si verificano nei magazzini. I prezzi delle polizze contro i furti sono legati alla quantità media giornaliera di prodotti presenti nei magazzini e al loro valore. Esistono diverse tipologie di polizze e la dirigenza vuole capire quali sono le polizze più adeguate per i propri magazzini analizzando quali modelli di **prodotti** e in quali quantità sono mediamente presenti a livello giornaliero nei magazzini. Alcune polizze sono specifiche per un singolo magazzino mentre altre sono relative ad insiemi di magazzini che si trovano nella stessa città, regione o provincia. In particolare, i premi assicurativi delle polizze sono calcolati considerando, su base media giornaliera:
- il ***numero di prodotti*** presenti nei magazzini per ogni modello di prodotto presente
- il ***valore dei prodotti*** presenti nei magazzini per ogni modello di prodotti.
Viste le problematiche appena descritte la ditta ha deciso di creare un apposito data warehouse per gestire le informazioni relative all’**uso dei magazzini** e al loro contenuto.
Relativamente all’analisi dell’uso dei magazzini (superficie utilizzata) la dirigenza è interessata ad analizzare la percentuale giornaliera di ***superficie libera**** rispetto alla ***superficie totale disponibile*** in ogni magazzino in funzione del magazzino e del luogo in cui esso si trova (*città, provincia, regione*). La superficie totale disponibile per ogni magazzino può variare di giorno in quanto alcune aree dei magazzini possono essere affittate a terzi o essere temporaneamente inagibili (queste aree non fanno parte della superficie totale disponibile). L’analisi deve essere possibile a livello di singola *data*, ma anche a livello di *mese, trimestre, quadrimestre, semestre e anno*.
Per stimare i costi delle possibili polizze assicurative la ditta deve poter analizzare il numero medio giornaliero di *prodotti presenti nei magazzini* e il loro valore complessivo giornaliero a livello di singola *data*, a livello di *mese, di trimestre, quadrimestre, di semestre e di anno*. Tali informazioni devono essere disponibili in funzione del *modello dei prodotti* e della loro *categoria* (scopa elettrica, impastatrice, ..), del magazzino e del luogo in cui si trova (*città, provincia, regione*).

#### Sono di seguito riportate alcune delle interrogazioni frequenti di interesse per la dirigenza della ditta:
1. Relativamente al primo trimestre dell’anno 2013, considerando solo i magazzini della città di Torino, trovare per ogni coppia (magazzino,data)il valore complessivo di prodotti presenti in tale data nel magazzino e il valore complessivo medio giornaliero di prodotti presenti nel magazzino nel corso della settimana precedente la data in esame (data in esame inclusa).
2. Relativamente all’anno 2014, trovare per ogni coppia(città,data) la percentuale di superficie libera giornaliera nella città. Associare ad ogni coppia un attributo di rank legato alla percentuale di superficie libera giornaliera nella città (1 per la coppia con la più bassa percentuale di superficie libera giornaliera).
3. Relativamente ai primi sei mesi dell’anno 2014, trovare per ogni coppia (magazzino,data) la percentuale di superficie libera giornaliera.
4. Relativamente all’anno 2013, trovare per ogni coppia (magazzino,mese) il valore complessivo medio giornaliero di prodotti presenti.
5. Relativamente all’anno 2015, trovare per ogni regione il valore complessivo medio giornaliero di prodotti presenti nella regione.
6. Relativamente all’anno 2014, trovare per ogni coppia(mese, regione) la percentuale di superficie libera giornaliera nella regione.


## Progettazione concettuale

### Facts:
- #### Uso Magazzini
- #### Prodotti

![Schema concettuale](../Data%20Warehouse/img/01_DSTBD_Progettazione%20Magazzini.png)

## Progettazione logica

**Uso_Magazzini** (*IdTempo*, *IdMagazzino*, SupLibera, SupTotale)

**Tempo** (*IdTempo*, Data, Mese, Trimestre, Quadrimestre, Semestre, Anno)

**Magazzino** (*IdMagazzino*, Magazzino, Città, Provincia, Regione)

----------------------------------------------------------------------------------

**Prodotti** (*IdTempo*, *IdMagazzino*, *IdModello*, NumProdotti, ValoreProdotti)

**Tempo** (*IdTempo*, Data, Mese, Trimestre, Quadrimestre, Semestre, Anno)

**Magazzino** (*IdMagazzino*, Magazzino, Città, Provincia, Regione)

**Modello** (*IdModello*, Modello, Categoria)


## Query SQL

1. Relativamente al primo trimestre dell’anno 2013, considerando solo i magazzini della città di Torino, trovare per ogni coppia (magazzino,data) il valore complessivo di prodotti presenti in tale data nel magazzino e il valore complessivo medio giornaliero di prodotti presenti nel magazzino nel corso della settimana precedente la data in esame (data in esame inclusa).

In [ ]:
SELECT M.Magazzino, T.Data, SUM(P.ValoreProdotti) AS ValoreComplessivo, 
    AVG(SUM(P.ValoreProdotti)) OVER (
        PARTITION BY M.Magazzino
        ORDER BY T.Data
        RANGE BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS ValoreMedio
FROM Prodotti P
JOIN Tempo T ON P.IdTempo = T.IdTempo
JOIN Magazzino M ON P.IdMagazzino = M.IdMagazzino
WHERE T.Anno = 2013 AND M.Città = 'Torino'
GROUP BY M.Magazzino, T.Data;

2. Relativamente all’anno 2014, trovare per ogni coppia(città,data) la percentuale di superficie libera giornaliera nella città. Associare ad ogni coppia un attributo di rank legato alla percentuale di superficie libera giornaliera nella città (1 per la coppia con la più bassa percentuale di superficie libera giornaliera).

In [ ]:
SELECT M.Città, T.Data, 100 * (UM.SupLibera/UM.SupTotale) AS PercSupLibera
    RANK() OVER (
        ORDER BY SUM(100 * (UM.SupLibera/UM.SupTotale))
    )
FROM Uso_Magazzini UM 
JOIN Tempo T ON T.IdTempo = UM.IdTempo
JOIN Magazzino M ON M.IdMagazzino = UM.IdMagazzino
WHERE T.Anno = 2014
GROUP BY M.Città, T.Data;


3. Relativamente ai primi sei mesi dell’anno 2014, trovare per ogni coppia (magazzino,data) la percentuale di superficie libera giornaliera.

In [ ]:
SELECT M.Magazzino, T.Data, 100 * (UM.SupLibera/UM.SupTotale) AS PercSupLibera
    OVER (
        ORDER BY T.Mese
        ROWS B
    )
FROM Uso_Magazzini UM 
JOIN Magazzino M ON M.IdMagazzino = UM.IdMagazzino
JOIN Tempo T ON T.IdTempo = UM.IdTempo
WHERE T.Anno = 2014
GROUP BY M.Magazzino, T.Data;

4. Relativamente all’anno 2013, trovare per ogni coppia (magazzino,mese) il valore complessivo medio giornaliero di prodotti presenti.

In [ ]:
SELECT DISTINCT M.Magazzino, T.Mese, 
    AVG(SUM(P.ValoreProdotti)) OVER(
        PARTITION BY T.Mese, M.Magazzino
    ) AS ValoreMedioGiornaliero
FROM Prodotti P 
JOIN Magazzino M ON M.IdMagazzino = P.IdMagazzino
JOIN Tempo T ON t.IdTempo = P.IdTempo
WHERE T.Anno = 2013
GROUP BY M.Magazzino, T.Mese;

5. Relativamente all’anno 2015, trovare per ogni regione il valore complessivo medio giornaliero di prodotti presenti nella regione.

In [ ]:
SELECT DISTINCT M.Regione, 
    AVG(SUM(P.ValoreProdotti)) OVER(
        PARTITION BY M.Regione
    ) AS ValoreMedioGiornaliero
FROM Prodotti P 
JOIN Magazzino M ON M.IdMagazzino = P.IdMagazzino
JOIN Tempo T ON T.IdTempo = P.IdTempo
WHERE T.Anno = 2015
GROUP BY M.Regione, T.Data;

6. Relativamente all’anno 2014, trovare per ogni coppia(mese, regione) la percentuale di superficie libera giornaliera nella regione.

In [ ]:
SELECT T.Mese, M.Regione, 100 * SUM(UM.SupLibera) / SUM(UM.SupTotale) AS PercSupLibera
FROM Uso_Magazzino UM 
JOIN Magazzino M ON M.IdMagazzino = UM.IdMagazzino
JOIN Tempo T ON T.IdTempo = UM.IdTempo
WHERE T.Anno = 2014
GROUP By T.Mese, M.Regione;